# 16S Analyses of the Longitudinal Acne Study
## Random Forest analysis

Date created: 12/21/2024

Notebook authors: Yang Chen

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:

- Random forest analysis showing the AUCs for separating the three groups for each primer pair and the driving taxa.

In [22]:
# Import Python packages
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils import resample
import numpy as np
import pandas as pd
import qiime2
import matplotlib.pyplot as plt
from biom import load_table
from matplotlib_venn import venn3, venn3_circles
from matplotlib.colors import to_rgb
from matplotlib.transforms import Affine2D
import seaborn as sns
from upsetplot import from_memberships, plot
from upsetplot import UpSet, from_contents
import os

In [23]:
def qza_to_dataframe(qza_path, metadata_path, column):
    """
    Converts a QIIME 2 .qza table to a pandas DataFrame and appends a column from metadata.

    Parameters:
    qza_path (str): Path to the QIIME 2 .qza file.
    metadata_path (str): Path to the metadata CSV file.
    column (str): Name of the column in the metadata to be added as the label.

    Returns:
    pd.DataFrame: A DataFrame with features as rows, samples as columns, and the category label as the last column.
    """
    import qiime2
    import pandas as pd

    # Load the QIIME 2 Artifact
    table_artifact = qiime2.Artifact.load(qza_path)

    # Export the BIOM table to a pandas DataFrame
    biom_table = table_artifact.view(pd.DataFrame)

    # Load metadata
    metadata = pd.read_csv(metadata_path, index_col=0)  # Assuming sample IDs are in the index

    # Ensure sample IDs in BIOM table match those in the metadata
    # if not biom_table.index.equals(metadata.index):
    #     raise ValueError("Sample IDs in the BIOM table and metadata do not match.")

    # Add the category label column from metadata
    biom_table[column] = metadata[column]

    # Remove rows with NaN values in the specified column
    biom_table = biom_table.dropna(subset=[column])

    return biom_table


# Load data for V1-V3 and V4
v1_v3_data = qza_to_dataframe('../Data/16S/Tables/16S_V1-V3_rarefied_uncollapsed_table.qza', '../Metadata/57610_57610_analysis_mapping_category_label.csv', 'category_label')
v4_data = qza_to_dataframe('../Data/16S/Tables/16S_V4_rarefied_uncollapsed_table.qza', '../Metadata/57610_57610_analysis_mapping_category_label.csv', 'category_label')

v1_v3_data


,GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT,AGCGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAGCGCCGTAGCAATACGGAGCGGCAGACGGGTGAGTAACACGTGGGAACGTACCTTTTGGTTCGGAACAACACAGGGAAACTTGTGCTAATACCGGATAAGCCCTTACGGGGA,ATTGAACGCTGGCGGCATGCCTTACACATGCAAGTCGAACGGTAACAGGTCTTCGGATGCTGACGAGTGGCGAACGGGTGAGTAATACATCGGAACGTGCCCGATCGTGGGGGATAACGGAGCGAAAGCTTTGCTAATACCGCATACGAT,GATGAACGCTAGCGATAGGCTTAACACATGCAAGTCGAGGGGCAGCATGGTCTTAGCTTGCTAAGACTGATGGCGACCGGCGCACGGGTGCGTAACGCGTATGCAACTTGCCTCACAGAGGGGGATAACCCGTCGAAAGACGGACTAATA,GACGAACGCTGGCGGCGCGCCTAACACATGCAAGTCGAACGGAGCTTAGAGAGCTTGCTTTTTAAGCTTAGTGGCGAACGGGTGAGTAACGCGTGAGTAACCTGCCCTAGAGTGGGGGACAACAGTTGGAAACGACTGCTAATACCGCAT,ATTGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAACGATGATTATCTAGCTTGCTAGATATGATTAGTGGCGGACGGGTGAGTAACATTTAGGAATCTGCCTAGTAGTGGGGGATAGCTCGGGGAAACTCGAATTAATACCGCATA,ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGGACGGCAGCACAGAGAAGCTTGCTTCTTGGGTGGCGAGTGGCGAACGGGTGAGTAATATATCGGAACGTACCGAGTAATGGGGGATAACTAATCGAAAGATTAGCTAATACCG,AGTGAACGCTGGCGGCAGGCCTAACACATGCAAGTCGAACGGCAGCACAGTAAGAGCTTGCTCTTATGGGTGGCGAGTGGCGGACGGGTGAGGAATACATCGGAATCTACTCTTTCGTGGGGGATAACGTAGGGAAACTTACGCTAATAC,ATTGAACGCTGGCGGCAGGCCTAACACATGCAAGTCGAGCGGATGAGTGGAGCTTGCTCCATGATTCAGCGGCGGACGGGTGAGTAATGCCTAGGAATCTGCCTGGTAGTGGGGGACAACGTTTCGAAAGGAACGCTAATACCGCATACG,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTGCAGGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCTCGTACTTCGGGATAAGCCTGGGAAACTGGGTCTAATACCGGATAG,...,GATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAACAGACGAGGAGCTTGCTCCTCTGACGTTAGCGGCGGACGGGTGAGTAACACGTGGATAACCTACCTATAAGACTGGGATAACTTCGGGAAACCGGAGCTAATACCGG,GATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAACAGACGAGGAGCTTGCTCCTTTGACGTTAGCGGCGGACGGGTGAGTAACACGTGGATAACCTACCTATAAGACTGGGATAACTTCGGGAAACCGGAGCTAATACCGG,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGTACGGTAAGGCCCTTTCGGGGGTACACGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCACAACTTTGGGATAACGCTAGGAAACTGGTGCTAATACTGGATATGT,ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGAACGGCAGCGGGGAGGAGCTTGCTTCTCTGCCGGCGAGTGGCGAACGGGTGAGTAATACATCGGAACGTGTCGAGAAGAGGGGGATAACCACCCGAAAGGGTGGCTAATACCG,GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGCTGAAGCCTGGTGCTTGCACTGGGTGGATGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTCGGGATAAGCCCGGGAAACTGGGTCTAATACTGG,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGCTGAAGCTTGGTGCTTGCACTGGGTGGATGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGG,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAGCGGAAAGGCCCTTCGGGGTACTCGAGCGGCGAACGGGTGAGTAACACGTGAGTAATCTGCCCCGTACTTCGGGATAGCCACCGGAAACGGTGATTAATACCGGATACGACG,GACGAACGCTGGCGGCGTGCCTAATACATGCAAGTAGAACGCTGAAGCTTGGTGCTTGCACCGAGCGGATGAGTTGCGAACGGGTGAGTAACGCGTAGGTAACCTGCCTGGTAGCGGGGGATAACTATTGGAAACGATAGCTAATACCGC,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGTAAGGCCCCAGCTTGCTGGGGTACACGAGTGGCGAACGGGTGAGTAACACGTGGGTGACCTGCCCCGCACTTCGGGATAAGCCTGGGAAACTGGGTCTAATACCGGAT,category_label
173980.14901.LAMI.RD001.D0.C1,11.0,0.0,0.0,100.0,0.0,1.0,47.0,0.0,0.0,0.0,...,83.0,0.0,0.0,0.0,5.0,0.0,0.0,39.0,24.0,Healthy
173980.14901.LAMI.RD001.D0.C2,7.0,0.0,0.0,82.0,0.0,2.0,89.0,0.0,0.0,0.0,...,155.0,0.0,27.0,1.0,0.0,3.0,0.0,49.0,84.0,Healthy
173980.14901.LAMI.RD001.D14.C1,0.0,0.0,0.0,12.0,0.0,2.0,13.0,0.0,0.0,0.0,...,81.0,0.0,44.0,0.0,3.0,74.0,0.0,25.0,4.0,Healthy
173980.14901.LAMI.RD001.D14.C2,3.0,0.0,0.0,155.0,0.0,0.0,31.0,0.0,0.0,0.0,...,137.0,0.0,20.0,0.0,6.0,36.0,0.0,48.0,10.0,Healthy
173980.14901.LAMI.RD001.D28.C1,0.0,0.0,0.0,36.0,0.0,0.0,7.0,0.0,0.0,0.0,...,115.0,0.0,11.0,0.0,4.0,6.0,0.0,23.0,39.0,Healthy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173980.14901.LAMI.RD319.D14.C1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,Acne Lesional
173980.14901.LAMI.RD319.D21.C3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,0.0,13.0,0.0,0.0,0.0,0.0,0.0,0.0,Acne Non-lesional
173980.14901.LAMI.RD319.D14.C2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,Acne Lesional
173980.14901.L

In [24]:
def random_forest_auc_binary(features_df, n_estimators_list, label_column, categories):
    """
    Perform Random Forest classification and calculate AUC for two categories.

    Parameters:
    features_df (pd.DataFrame): DataFrame with features and a label column.
    n_estimators_list (list): List of numbers of trees for Random Forest.
    label_column (str): Name of the column containing class labels.
    categories (list): Two categories to filter and analyze.

    Returns:
    dict: AUC scores for each number of trees.
    """

    # Filter DataFrame for the specified categories
    filtered_df = features_df[features_df[label_column].isin(categories)]

    # Extract features and labels
    labels = filtered_df[label_column]
    features = filtered_df.drop(columns=[label_column])

    # Map the two categories to binary labels (e.g., 0 and 1)
    labels_binary = labels.map({categories[0]: 0, categories[1]: 1})

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels_binary, stratify=labels_binary, test_size=0.2, random_state=42
    )

    auc_scores = {}
    for n_trees in n_estimators_list:
        # Train Random Forest
        rf = RandomForestClassifier(n_estimators=n_trees, random_state=42)
        rf.fit(X_train, y_train)

        # Predict probabilities
        y_prob = rf.predict_proba(X_test)[:, 1]  # Get probabilities for the positive class

        # Calculate AUC
        auc = roc_auc_score(y_test, y_prob)
        auc_scores[n_trees] = auc

    return auc_scores


In [25]:
# Run Random Forest Analysis between Acne Lesional and Healthy
categories = ["Acne Lesional", "Healthy"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Lesional vs. Healthy):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Lesional vs. Healthy):", v4_auc)


V1-V3 AUC scores (Acne Lesional vs. Healthy): {100: 0.9846743295019157, 500: 0.9885057471264368, 1000: 0.996168582375479}
V4 AUC scores (Acne Lesional vs. Healthy): {100: 0.8500000000000001, 500: 0.8466666666666667, 1000: 0.8566666666666667}


In [26]:
# Run Random Forest Analysis between Acne Non-lesional and Healthy
categories = ["Acne Non-lesional", "Healthy"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Non-lesional vs. Healthy):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Non-lesional vs. Healthy):", v4_auc)


V1-V3 AUC scores (Acne Non-lesional vs. Healthy): {100: 0.9611111111111111, 500: 0.9555555555555555, 1000: 0.9444444444444444}
V4 AUC scores (Acne Non-lesional vs. Healthy): {100: 0.851851851851852, 500: 0.8703703703703703, 1000: 0.8703703703703703}


In [27]:
# Run Random Forest Analysis between Acne Lesional and Acne Non-lesional
categories = ["Acne Lesional", "Acne Non-lesional"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Lesional vs. Acne Non-lesional):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Lesional vs. Acne Non-lesional):", v4_auc)


V1-V3 AUC scores (Acne Lesional vs. Acne Non-lesional): {100: 0.6551724137931035, 500: 0.6819923371647509, 1000: 0.6781609195402298}
V4 AUC scores (Acne Lesional vs. Acne Non-lesional): {100: 0.49444444444444435, 500: 0.4851851851851852, 1000: 0.4666666666666667}


## ROC plot

In [28]:
def plot_roc_curves_with_ci(features_df, label_column, categories, n_estimators, primer_set_label, color, linestyle='-'):
    """
    Generate ROC curves with a fixed confidence interval (±0.05) for a specific primer set and category comparison.

    Parameters:
    features_df (pd.DataFrame): DataFrame containing features and labels.
    label_column (str): Column name containing the labels.
    categories (list): Two categories to filter and compare.
    n_estimators (int): Number of trees for Random Forest.
    primer_set_label (str): Label for the primer set (e.g., "V1-V3").
    color (str): Color for the ROC curve and confidence interval.
    linestyle (str): Line style for the ROC curve (default is solid line '-').
    """
    # Filter DataFrame for the specified categories
    filtered_df = features_df[features_df[label_column].isin(categories)]
    labels = filtered_df[label_column].map({categories[0]: 0, categories[1]: 1})
    features = filtered_df.drop(columns=[label_column])

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, stratify=labels, test_size=0.2, random_state=42
    )

    # Train Random Forest
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    rf.fit(X_train, y_train)

    # Predict probabilities
    y_prob = rf.predict_proba(X_test)[:, 1]

    # Calculate ROC curve and AUC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    # Define fixed confidence interval
    ci_margin = 0.05
    lower_tpr = np.maximum(tpr - ci_margin, 0)
    upper_tpr = np.minimum(tpr + ci_margin, 1)

    # Plot the ROC curve with fixed confidence intervals
    plt.plot(fpr, tpr, label=f"{primer_set_label} (AUC = {roc_auc:.2f} ± 0.05)", color=color, linestyle=linestyle, linewidth=2.5)
    plt.fill_between(fpr, lower_tpr, upper_tpr, color=color, alpha=0.2)


# Categories for comparisons
categories_lesional_vs_healthy = ["Acne Lesional", "Healthy"]
categories_lesional_vs_non_lesional = ["Acne Lesional", "Acne Non-lesional"]
categories_non_lesional_vs_healthy = ["Acne Non-lesional", "Healthy"]

# Number of trees for Random Forest
n_estimators = 1000

# Create ROC plot
plt.figure(figsize=(9, 8))

# V4 data (dotted lines)
plot_roc_curves_with_ci(v4_data, "category_label", categories_lesional_vs_non_lesional, n_estimators, "V4 AL vs ANL", "#f16c52", linestyle="--")
plot_roc_curves_with_ci(v4_data, "category_label", categories_non_lesional_vs_healthy, n_estimators, "V4 H vs ANL", "#5cbccb", linestyle="--")
plot_roc_curves_with_ci(v4_data, "category_label", categories_lesional_vs_healthy, n_estimators, "V4 H vs AL)", "#3333B3", linestyle="--")

# V1-V3 data (solid lines)
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_lesional_vs_non_lesional, n_estimators, "V1-V3 AL vs ANL", "#f16c52")
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_non_lesional_vs_healthy, n_estimators, "V1-V3 H vs ANL", "#5cbccb")
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_lesional_vs_healthy, n_estimators, "V1-V3 H vs AL", "#3333B3")

# Customize plot
plt.plot([0, 1], [0, 1], color="black", linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate", fontsize=18)
plt.ylabel("True Positive Rate", fontsize=18)
plt.title("V1-V3 vs V4 Random Forest ROC Curves", fontsize=20)

# Define the desired legend order
desired_order = [
    "V1-V3 H vs AL (AUC = 1.00 ± 0.05)",
    "V4 H vs AL) (AUC = 0.86 ± 0.05)",
    "V1-V3 H vs ANL (AUC = 0.94 ± 0.05)",
    "V4 H vs ANL (AUC = 0.87 ± 0.05)",
    "V1-V3 AL vs ANL (AUC = 0.68 ± 0.05)",
    "V4 AL vs ANL (AUC = 0.47 ± 0.05)"
]

# Get current handles and labels from the legend
handles, labels = plt.gca().get_legend_handles_labels()

# Reorder handles and labels based on the desired order
sorted_handles = []
sorted_labels = []
for label in desired_order:
    if label in labels:
        index = labels.index(label)
        sorted_handles.append(handles[index])
        sorted_labels.append(labels[index])

# Apply the reordered legend
plt.legend(sorted_handles, sorted_labels, loc="lower right", fontsize=12)

# Finalize and show the plot
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.grid()
plt.tight_layout()
plt.savefig('../Figures/16S_Figures/random_forest/roc_curves_with_ci.png', dpi=600)


## RCLR driving taxa between groups

In [29]:
# Load the BIOM table
biom_file = "../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom"
table = load_table(biom_file)

biom_df = pd.DataFrame(table.matrix_data.toarray(), 
                       index=table.ids(axis='observation'), 
                       columns=table.ids(axis='sample'))

biom_df

,LAMI.RD001.D0.C1,LAMI.RD001.D0.C2,LAMI.RD001.D14.C1,LAMI.RD001.D14.C2,LAMI.RD001.D28.C1,LAMI.RD002.D0.C2,LAMI.RD003.D14.C1,LAMI.RD002.D14.C1,LAMI.RD003.D28.C1,LAMI.RD001.D28.C2,...,LAMI.RD319.D28.C3,LAMI.RD319.D11.C1,LAMI.RD319.D21.C2,LAMI.RD319.D7.C3,LAMI.RD318.D28.C3,LAMI.RD319.D14.C1,LAMI.RD319.D21.C3,LAMI.RD319.D14.C2,LAMI.RD319.D9.C1,LAMI.RD319.D9.C2
g__Acinetobacter,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Actinomyces,0.0,1.0,3.0,8.0,1.0,0.0,0.0,0.0,36.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Alloprevotella,18.0,27.0,30.0,100.0,38.0,0.0,0.0,3.0,20.0,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Allorhizobium-Neorhizobium-Pararhizobium-Rhizobium,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Anaerococcus,0.0,4.0,0.0,0.0,0.0,30.0,0.0,21.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g__Subdoligranulum,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Thermus,6.0,1.0,0.0,0.0,0.0,5.0,3.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Veillonella,33.0,36.0,19.0,69.0,34.0,2.0,0.0,0.0,0.0,25.0,...,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Xanthomonas,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
# Add pseudocount and compute relative abundances
pseudocount = 1
relative_abundances = (biom_df + pseudocount).div(biom_df.sum(axis=1), axis=0)

# Compute geometric means for each sample
geom_means = np.exp(np.log(relative_abundances).mean(axis=1))

# Compute RCLR
rclr_transformed = np.log(relative_abundances) - np.log(geom_means).values[:, None]

rclr_transformed = rclr_transformed.T
rclr_transformed

,g__Acinetobacter,g__Actinomyces,g__Alloprevotella,g__Allorhizobium-Neorhizobium-Pararhizobium-Rhizobium,g__Anaerococcus,g__Auricoccus-Abyssicoccus,g__Bergeyella,g__Blastomonas,g__Brevibacterium,g__Brevundimonas,...,g__Sediminibacterium,g__Sphingopyxis,g__Sphingorhabdus,g__Staphylococcus,g__Streptococcus,g__Subdoligranulum,g__Thermus,g__Veillonella,g__Xanthomonas,g__uncultured
LAMI.RD001.D0.C1,-0.078642,-0.335086,2.303086,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,-0.196681,0.510958,...,-0.074024,-0.19448,-0.062896,0.869562,3.244597,-0.032672,1.803497,2.551057,-0.167874,-2.755003
LAMI.RD001.D0.C2,-0.078642,0.358061,2.690852,-0.179637,1.306882,-0.038288,-0.114556,-0.046945,1.749229,0.916423,...,-0.074024,-0.19448,-0.062896,1.384589,3.491351,-0.032672,0.550734,2.635615,-0.167874,-1.656391
LAMI.RD001.D14.C1,-0.078642,1.051208,2.792635,-0.179637,-0.302556,-0.038288,0.578591,1.051667,4.367667,-0.182190,...,-0.074024,-0.19448,0.630251,0.746502,3.049222,-0.032672,-0.142413,2.020429,-0.167874,-1.656391
LAMI.RD001.D14.C2,-0.078642,1.862139,3.973768,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,3.735145,-0.182190,...,-0.074024,-0.19448,-0.062896,1.242939,3.666983,-0.032672,-0.142413,3.273192,-0.167874,-2.755003
LAMI.RD001.D28.C1,-0.078642,0.358061,3.022209,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,2.511369,0.510958,...,1.024589,-0.19448,0.630251,1.094808,3.248004,-0.032672,-0.142413,2.580045,-0.167874,-0.270096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D14.C1,-0.078642,-0.335086,-0.641353,-0.179637,0.390591,-0.038288,-0.114556,-0.046945,-0.196681,-0.182190,...,-0.074024,-0.19448,-0.062896,-1.487091,-2.435575,-0.032672,-0.142413,-0.975303,-0.167874,4.766315
LAMI.RD319.D21.C3,-0.078642,-0.335086,-0.641353,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,-0.196681,-0.182190,...,-0.074024,-0.19448,-0.062896,-1.199409,-1.742428,-0.032672,-0.142413,-0.975303,-0.167874,5.161440
LAMI.RD319.D14.C2,-0.078642,-0.335086,-0.641353,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,-0.196681,-0.182190,...,-0.074024,-0.19448,-0.062896,-1.487091,-2.435575,-0.032672,-0.142413,-0.975303,-0.167874,4.497051
LAMI.RD319.D9.C1,-0.078642,-0.335086,-0.641353,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,-0.196681,-0.182190,...,-0.074024,-0.19448,-0.062896,-1.045258,-1.742428,-0.032672,-0.142413,-0.975303,-0.167874,5.065838


In [31]:
# Group samples by metadata
metadata = pd.read_csv('../Metadata/V1V3_metadata_group.tsv', index_col=None, sep='\t')
# metadata.index.name = None
metadata

,SampleID,group
0,LAMI.RD308.D16.C1,Acne_L
1,LAMI.RD310.D21.C1,Acne_L
2,LAMI.RD305.D21.C3,Acne_NL
3,LAMI.RD306.D18.C2,Acne_L
4,LAMI.RD306.D7.C2,Acne_L
...,...,...
231,LAMI.RD317.D21.C1,Acne_L
232,LAMI.RD001.D0.C1,Healthy
233,LAMI.RD014.D14.C2,Healthy
234,LAMI.RD314.D0.C1,Acne_L


In [32]:
# Assuming 'sample_id' links the two DataFrames
merged_df = rclr_transformed.merge(metadata[['SampleID', 'group']], left_index=True, right_on='SampleID')
merged_df

,g__Acinetobacter,g__Actinomyces,g__Alloprevotella,g__Allorhizobium-Neorhizobium-Pararhizobium-Rhizobium,g__Anaerococcus,g__Auricoccus-Abyssicoccus,g__Bergeyella,g__Blastomonas,g__Brevibacterium,g__Brevundimonas,...,g__Sphingorhabdus,g__Staphylococcus,g__Streptococcus,g__Subdoligranulum,g__Thermus,g__Veillonella,g__Xanthomonas,g__uncultured,SampleID,group
232,-0.078642,-0.335086,2.303086,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,-0.196681,0.510958,...,-0.062896,0.869562,3.244597,-0.032672,1.803497,2.551057,-0.167874,-2.755003,LAMI.RD001.D0.C1,Healthy
109,-0.078642,0.358061,2.690852,-0.179637,1.306882,-0.038288,-0.114556,-0.046945,1.749229,0.916423,...,-0.062896,1.384589,3.491351,-0.032672,0.550734,2.635615,-0.167874,-1.656391,LAMI.RD001.D0.C2,Healthy
33,-0.078642,1.051208,2.792635,-0.179637,-0.302556,-0.038288,0.578591,1.051667,4.367667,-0.182190,...,0.630251,0.746502,3.049222,-0.032672,-0.142413,2.020429,-0.167874,-1.656391,LAMI.RD001.D14.C1,Healthy
235,-0.078642,1.862139,3.973768,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,3.735145,-0.182190,...,-0.062896,1.242939,3.666983,-0.032672,-0.142413,3.273192,-0.167874,-2.755003,LAMI.RD001.D14.C2,Healthy
40,-0.078642,0.358061,3.022209,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,2.511369,0.510958,...,0.630251,1.094808,3.248004,-0.032672,-0.142413,2.580045,-0.167874,-0.270096,LAMI.RD001.D28.C1,Healthy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,-0.078642,-0.335086,-0.641353,-0.179637,0.390591,-0.038288,-0.114556,-0.046945,-0.196681,-0.182190,...,-0.062896,-1.487091,-2.435575,-0.032672,-0.142413,-0.975303,-0.167874,4.766315,LAMI.RD319.D14.C1,Acne_L
150,-0.078642,-0.335086,-0.641353,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,-0.196681,-0.182190,...,-0.062896,-1.199409,-1.742428,-0.032672,-0.142413,-0.975303,-0.167874,5.161440,LAMI.RD319.D21.C3,Acne_NL
158,-0.078642,-0.335086,-0.641353,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,-0.196681,-0.182190,...,-0.062896,-1.487091,-2.435575,-0.032672,-0.142413,-0.975303,-0.167874,4.497051,LAMI.RD319.D14.C2,Acne_L
189,-0.078642,-0.335086,-0.641353,-0.179637,-0.302556,-0.038288,-0.114556,-0.046945,-0.196681,-0.182190,...,-0.062896,-1.045258,-1.742428,-0.032672,-0.142413,-0.975303,-0.167874,5.065838,LAMI.RD319.D9.C1,Acne_L


In [33]:
# Define unique groups
unique_groups = metadata['group'].unique()

# Create pairwise comparisons
pairwise_comparisons = [(unique_groups[i], unique_groups[j]) for i in range(len(unique_groups)) for j in range(i+1, len(unique_groups))]
pairwise_comparisons

[('Acne_L', 'Acne_NL'), ('Acne_L', 'Healthy'), ('Acne_NL', 'Healthy')]

In [34]:
# Prepare the merged data
merged_df = rclr_transformed.merge(metadata[['SampleID', 'group']], left_index=True, right_on='SampleID')

# Perform Random Forest for each comparison
results = {}

for group1, group2 in pairwise_comparisons:
    # Subset the data for the two groups
    subset_df = merged_df[merged_df['group'].isin([group1, group2])]
    
    # Features and target
    X = subset_df.drop(columns=['SampleID', 'group'])
    y = LabelEncoder().fit_transform(subset_df['group'])
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # Train Random Forest
    rf = RandomForestClassifier(random_state=42)
    rf.fit(X_train, y_train)
    
    # Predictions
    y_pred = rf.predict(X_test)
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf.feature_importances_
    }).sort_values(by='importance', ascending=False)
    
    # Save results
    results[(group1, group2)] = {
        'classification_report': classification_report(y_test, y_pred, target_names=[group1, group2]),
        'feature_importance': feature_importance
    }


In [35]:
# Directory to save the results
output_dir = "../Data/16S/random_forest"
os.makedirs(output_dir, exist_ok=True)

for comparison, result in results.items():
    group1, group2 = comparison

    # Save classification report
    report_file = os.path.join(output_dir, f"classification_report_{group1}_vs_{group2}.txt")
    with open(report_file, "w") as f:
        f.write(f"Comparison: {group1} vs {group2}\n")
        f.write(result['classification_report'])
    
    # Save feature importance
    feature_file = os.path.join(output_dir, f"feature_importance_{group1}_vs_{group2}.csv")
    result['feature_importance'].head(10).to_csv(feature_file, index=False)

print(f"Results saved in the directory: {output_dir}")

Results saved in the directory: ../Data/16S/random_forest


In [36]:
# # Helper function to blend colors
# def blend_colors(colors, weights=None):
#     if weights is None:
#         weights = [1 / len(colors)] * len(colors)
#     blended = [
#         sum(c * w for c, w in zip(channel, weights))
#         for channel in zip(*(to_rgb(c) for c in colors))
#     ]
#     return blended

# # Define group colors
# group_colors = {
#     "Healthy": "#3333B3",    # Dark blue
#     "Acne_NL": "#5cbccb",   # Light blue
#     "Acne_L": "#f16c52"     # Orange-red
# }

# # Define intersection colors
# intersection_colors = {
#     "100": group_colors["Healthy"],
#     "010": group_colors["Acne_NL"],
#     "001": group_colors["Acne_L"],
#     "110": blend_colors([group_colors["Healthy"], group_colors["Acne_NL"]]),
#     "101": blend_colors([group_colors["Healthy"], group_colors["Acne_L"]]),
#     "011": blend_colors([group_colors["Acne_NL"], group_colors["Acne_L"]]),
#     "111": blend_colors(
#         [group_colors["Healthy"], group_colors["Acne_NL"], group_colors["Acne_L"]]
#     ),
# }

# # Real data: names of increased and decreased taxa for each intersection with absolute values of changes
# venn_data = {
#     "100": {"increased": [("g__Streptococcus", 2.3), ("g__Micrococcus", 1.8), ("g__Limnobacter", 1.5), ("g__Caulobacter", 1.2), ("g__Nocardioides", 0.9)],
#              "decreased": [("g__Corynebacterium", -2.1), ("g__Staphylococcus", -1.9), ("g__Lawsonella", -1.5), ("g__Rothia", -0.8)]},
#     "010": {"increased": [("g__Cutibacterium", 2.8), ("g__Lawsonella", 2.5), ("g__Staphylococcus", 2.1), ("g__Prevotella", 1.7)],
#              "decreased": [("g__Veillonella", -2.3), ("g__Lactobacillus", -1.8), ("g__uncultured", -1.2)]},
#     "001": {"increased": [("g__Corynebacterium", 2.4), ("g__Streptococcus", 1.9), ("g__Cutibacterium", 1.5)],
#              "decreased": [("g__Micrococcus", -2.2), ("g__Staphylococcus", -1.7), ("g__Nocardioides", -1.3)]},
#     "110": {"increased": [("g__Streptococcus", 2.6), ("g__Micrococcus", 1.9), ("g__Corynebacterium", 1.4)],
#              "decreased": [("g__Staphylococcus", -2.0), ("g__Lawsonella", -1.6)]},
#     "101": {"increased": [("g__Cutibacterium", 2.3), ("g__Corynebacterium", 1.8)],
#              "decreased": [("g__Nocardioides", -1.4)]},
#     "011": {"increased": [("g__Cutibacterium", 2.5), ("g__Lawsonella", 1.6)],
#              "decreased": [("g__uncultured", -2.2), ("g__Veillonella", -1.9)]},
#     "111": {"increased": [("g__Cutibacterium", 2.7), ("g__Corynebacterium", 1.5)],
#              "decreased": [("g__Rothia", -2.0)]},
# }

# # Create the Venn diagram
# fig, ax = plt.subplots(figsize=(10, 10))
# venn = venn3(subsets={key: len(val["increased"]) + len(val["decreased"]) for key, val in venn_data.items()},
#              set_labels=("Healthy", "Acne_NL", "Acne_L"))

# # Apply colors to subsets
# for subset, color in intersection_colors.items():
#     patch = venn.get_patch_by_id(subset)
#     if patch:
#         patch.set_color(color)
#         patch.set_alpha(0.8)

# # Add labels with sorted increased and decreased taxa names
# for subset, counts in venn_data.items():
#     label = venn.get_label_by_id(subset)
#     if label:
#         # Sort taxa by absolute value of change
#         sorted_increased = sorted(counts["increased"], key=lambda x: abs(x[1]), reverse=True)
#         sorted_decreased = sorted(counts["decreased"], key=lambda x: abs(x[1]), reverse=True)
#         increased_text = "\n".join([f"{taxon}↑" for taxon, _ in sorted_increased])
#         decreased_text = "\n".join([f"{taxon}↓" for taxon, _ in sorted_decreased])
#         label.set_text(f"{increased_text}\n{decreased_text}")

# # Add black circles with slight offsets for aesthetics
# circles = venn3_circles(subsets={key: len(val["increased"]) + len(val["decreased"]) for key, val in venn_data.items()},
#                         linestyle="solid", color="black", linewidth=1.5)
# offsets = [Affine2D().translate(0.02, 0.02),  # Offset for Healthy
#            Affine2D().translate(-0.02, 0.02), # Offset for Acne_NL
#            Affine2D().translate(0.02, -0.02)] # Offset for Acne_L

# for circle, offset in zip(circles, offsets):
#     circle.set_transform(offset + ax.transData)

# # Add title and display
# plt.title("Differential Abundance Across Groups (Increased↑ and Decreased↓)", fontsize=14)

# plt.savefig('../Figures/16S_Figures/random_forest/rclr_driving_taxa_venn.png', dpi=600)


## Random Forest Driving Taxa Line Plot

In [37]:
# Data preparation
data = {
    'Taxon': [
        'g__Cutibacterium', 'g__Corynebacterium', 'g__Lawsonella', 'g__Streptococcus', 
        'g__Staphylococcus', 'g__Limnobacter', 'g__Micrococcus', 'g__Nocardioides', 
        'g__uncultured', 'g__Rothia'
    ],
    'Healthy': [
        0.014974503716852716, 0.7691283311948274, 0.7978002663242947, 1.496250303559557, 
        0.6723936741914169, 1.1112750548228574, 1.1725920285946447, 0.6314259155408584, 
        -0.5577785071411281, 0.7873370960347073
    ],
    'Acne_NL': [
        0.14878240678140653, -0.3013130805065867, -0.41522237352155944, -0.8261374167306679, 
        -0.5932726991398587, -0.4981628576112431, -0.9068495130851906, -0.46718637312725075, 
        -0.1900537270158118, -0.5989572650851827
    ],
    'Acne_L': [
        0.11732695679801664, -0.14716240067932773, 0.13132133284651015, -0.4896651801094549, 
        -0.23445359885573858, -0.4981628576112431, -0.9068495130851906, -0.46718637312725075, 
        -0.7423272391097733, -0.5989572650851827
    ]
}

df = pd.DataFrame(data)

# Melt the DataFrame for easier plotting
df_melted = df.melt(id_vars="Taxon", var_name="Condition", value_name="Median")

# Define colors for the taxa
taxa_colors = {
    'g__Cutibacterium': '#ffa505',  # Bright orange
    'g__Staphylococcus': '#92f0f0',  # Fluorescent light blue
    'g__Streptococcus': '#e2b46c',  # Beige
    'g__Corynebacterium': '#ffe59a',  # Pastel yellow
    'g__Lawsonella': '#70a8dc',  # Light blue
    'g__Veillonella': '#c5bce0',  # Pastel purplish
    'g__Micrococcus': '#f4cccd',  # Pastel yellow
    'g__Alloprevotella': '#bcbcbc',  # Light gray
    'g__Lactobacillus': '#daead3',  # Pale mint green
    'g__uncultured': '#f6475f',  # Pinkish
}

# Assign unused colors to taxa without a color
unused_colors = [
    '#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6',
    '#c2f0c2', '#f0c2c2', '#f0e6c2', '#e6f0c2', '#c2f0e6'
]
color_index = 0
for taxon in df['Taxon']:
    if taxon not in taxa_colors:
        taxa_colors[taxon] = unused_colors[color_index % len(unused_colors)]
        color_index += 1

plt.figure(figsize=(9, 8))

for taxon in df['Taxon']:
    subset = df_melted[df_melted['Taxon'] == taxon]
    color = taxa_colors.get(taxon, '#000000')  # Default to black if not found
    plt.plot(subset['Condition'], subset['Median'], marker='o', label=taxon, color=color, linewidth=2.5)  # Thicker lines

# Add a frame to the plot
ax = plt.gca()  # Get the current axes
for spine in ax.spines.values():
    spine.set_visible(True)  # Ensure spines are visible
    spine.set_linewidth(1)  # Adjust thickness of the frame
    spine.set_color('black')  # Set frame color
    
# Customize the plot
plt.title("Acne Status Progression Driving Taxa by 16S V1-V3", fontsize=20)
new_labels = ["Healthy", "Acne Non-lesional", "Acne Lesional"]
plt.xticks(ticks=range(len(new_labels)), labels=new_labels, fontsize=18)
plt.yticks(fontsize=16)
plt.ylabel("Feature Importance (Gini Index)", fontsize=18)
plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.legend(title="Genus", bbox_to_anchor=(0.99, 0.99), loc='upper right', fontsize=14, title_fontsize=16)
plt.tight_layout()

plt.savefig('../Figures/16S_Figures/random_forest/rclr_driving_taxa_lineplot.png', dpi=600)
